# Figure 3 analysis and plotting workflow

Run the sections in order. The first section performs the joint NMF analysis and saves all required tables; later sections read those outputs and generate the retained figures.


# 1. Group-balanced joint NMF analysis and table export


# Group-Balanced Joint NMF CM Decomposition

This notebook implements a new joint-training version of the nonepi CM decomposition. It is based on the same data-processing/NMF framework as the previous notebooks, but it does not reuse any previous hCM/sharedCM/tCM result.

What is inherited from the previous code:

- Read the same annotated AnnData: `adata_anno_cell_subtype_re.h5ad`.
- Keep non-epithelial cells with `cell_type != "Epi"`.
- Build a sample x `cell_subtype` fraction matrix.
- Use NMF to learn module subtype loadings.
- Use NNLS to estimate sample-level module usage from a fixed basis.
- Save module loadings, usage, rank-selection metrics, top subtype tables, and plot-ready matrices.

What is newly designed here:

- Normal-like and tumor samples are trained together from scratch.
- The model is group-balanced so normal-like and tumor groups have equal total weight, even though tumor has more samples.
- The model does not start from normal-derived hCM and does not read `hcm_tcm_outputs`.
- Modules are classified only after training according to tumor/normal usage pattern.
- The final module classes are `sharedCM`, `tCM`, and `normalCM`; output module names append the class suffix, for example `joint_07_tCM`.

Workflow:

1. Build a sample x non-epithelial `cell_subtype` fraction matrix.
2. Give the normal-like group and tumor group equal total weight during NMF.
3. Fit joint NMF from scratch and select total rank over `k=2..20` unless `module_k` is forced.
4. Refit all samples to the fixed joint basis by NNLS.
5. Classify each learned module as `sharedCM`, `tCM`, or `normalCM` based on tumor/normal usage ratio.

Outputs are written to `balanced_joint_nmf_outputs/`.


In [ ]:
#!/usr/bin/env python
"""
Group-balanced joint NMF for nonepi CM decomposition.

This notebook does not reuse the previous hCM/sharedCM/tCM results. It trains a
new joint model from normal-like and tumor nonepi samples together, while giving
the two groups equal total weight.

After fitting, modules are classified by their usage pattern:
- sharedCM: reused by both normal-like and tumor samples.
- tCM: tumor-enriched module.
- normalCM: normal-like-enriched module.
"""

from __future__ import annotations

import json
import math
import shutil
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.optimize import linear_sum_assignment, nnls
from scipy.stats import mannwhitneyu, norm, pearsonr, spearmanr
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity


# -----------------------------------------------------------------------------
# User-editable parameters
# -----------------------------------------------------------------------------


@dataclass(frozen=True)
class Config:
    project_dir: str = "/mnt/disk18t/lr_xcy/riku/codex_reseach/CM_analysis_2/cm_epi_analysis"
    input_h5ad: str = "adata_anno_cell_subtype_re.h5ad"
    output_dir: str = "balanced_joint_nmf_outputs"

    sample_col: str = "sample"
    status_col: str = "status"
    celltype_col: str = "cell_type"
    subtype_col: str = "cell_subtype"

    epi_label: str = "Epi"
    normal_status: str = "normal-like"
    tumor_status: str = "tumor"
    min_nonepi_cells_per_sample: int = 50

    # Set module_k to an integer to force total joint NMF rank.
    # Keep None to evaluate module_k_range and select automatically.
    module_k: int | None = None
    module_k_range: tuple[int, int] = (2, 20)
    rank_selection_seeds: tuple[int, ...] = (0, 1, 2, 3, 4)
    random_state: int = 0

    nmf_max_iter: int = 3000
    nmf_tol: float = 1e-5
    nmf_alpha_w: float = 0.0
    nmf_alpha_h: float = 1e-3
    nmf_l1_ratio: float = 0.1

    # Group balancing: each status group contributes this total weight to NMF.
    normal_group_total_weight: float = 0.5
    tumor_group_total_weight: float = 0.5

    # Module class rules based on tumor/normal mean usage ratio.
    # ratio <= normal_specific_max_ratio -> normalCM
    # ratio >= tumor_specific_min_ratio -> tCM
    # otherwise -> sharedCM
    normal_specific_max_ratio: float = 0.5
    tumor_specific_min_ratio: float = 2.0
    min_active_fraction_for_specific: float = 0.05
    force_shared_modules: tuple[str, ...] = ()
    force_tcm_modules: tuple[str, ...] = ()
    force_normalcm_modules: tuple[str, ...] = ()

    top_n_subtypes: int = 20
    top_n_nodes: int = 10
    edge_r_threshold: float = 0.25


CFG = Config()


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------


def ensure_dirs(base: Path) -> dict[str, Path]:
    joint_cm = base / "joint_cm"
    tables = joint_cm / "tables"
    dirs = {
        "base": base,
        "shared": base / "shared",
        "rank_selection": tables,
        "modules": tables,
        "module_groups": tables,
        "combined": tables,
        "summary": tables,
        "stats": tables,
        "mwu": joint_cm / "mwu",
        "networks": joint_cm / "networks",
        "epi_cm_coupling": joint_cm / "epi_cm_coupling",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def write_json(obj: dict, path: Path) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(obj, handle, indent=2, ensure_ascii=False)






def row_normalize(matrix: np.ndarray) -> np.ndarray:
    denom = matrix.sum(axis=1, keepdims=True)
    return np.divide(matrix, denom, out=np.zeros_like(matrix, dtype=float), where=denom != 0)


def explained_fraction(x: np.ndarray, recon: np.ndarray) -> float:
    denom = float(np.square(x).sum())
    if denom == 0:
        return float("nan")
    return 1.0 - float(np.square(x - recon).sum()) / denom


def explained_fraction_by_row(x: np.ndarray, recon: np.ndarray) -> np.ndarray:
    denom = np.square(x).sum(axis=1)
    sse = np.square(x - recon).sum(axis=1)
    out = np.full(x.shape[0], np.nan, dtype=float)
    np.divide(sse, denom, out=out, where=denom != 0)
    return 1.0 - out


def make_nmf(n_components: int, random_state: int, cfg: Config) -> NMF:
    return NMF(
        n_components=n_components,
        init="nndsvda",
        solver="cd",
        beta_loss="frobenius",
        random_state=random_state,
        max_iter=cfg.nmf_max_iter,
        tol=cfg.nmf_tol,
        alpha_W=cfg.nmf_alpha_w,
        alpha_H=cfg.nmf_alpha_h,
        l1_ratio=cfg.nmf_l1_ratio,
    )


def fit_nmf_best_seed(x: np.ndarray, n_components: int, seeds: Iterable[int], cfg: Config) -> tuple[NMF, np.ndarray, np.ndarray]:
    best_model: NMF | None = None
    best_w: np.ndarray | None = None
    best_h: np.ndarray | None = None
    best_error = math.inf

    for seed in seeds:
        model = make_nmf(n_components=n_components, random_state=seed, cfg=cfg)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=UserWarning)
            w = model.fit_transform(x)
        h = model.components_
        recon = w @ h
        error = float(np.square(x - recon).sum())
        if error < best_error:
            best_error = error
            best_model = model
            best_w = w
            best_h = h

    if best_model is None or best_w is None or best_h is None:
        raise RuntimeError("NMF failed for all seeds.")
    return best_model, best_w, best_h


def fit_nnls_usage(h: np.ndarray, x: np.ndarray) -> np.ndarray:
    design = h.T
    usage = np.zeros((x.shape[0], h.shape[0]), dtype=float)
    for i in range(x.shape[0]):
        usage[i, :] = nnls(design, x[i, :], maxiter=design.shape[1] * 10)[0]
    return usage


def matched_cosine_mean(h_a: np.ndarray, h_b: np.ndarray) -> float:
    sim = cosine_similarity(row_normalize(h_a), row_normalize(h_b))
    row_ind, col_ind = linear_sum_assignment(-sim)
    return float(sim[row_ind, col_ind].mean())


def stability_score(components: list[np.ndarray]) -> float:
    if len(components) < 2:
        return float("nan")
    scores: list[float] = []
    for i in range(len(components)):
        for j in range(i + 1, len(components)):
            scores.append(matched_cosine_mean(components[i], components[j]))
    return float(np.mean(scores)) if scores else float("nan")


# -----------------------------------------------------------------------------
# Data construction
# -----------------------------------------------------------------------------


def load_obs(input_h5ad: Path, cfg: Config) -> pd.DataFrame:
    adata = sc.read_h5ad(input_h5ad, backed="r")
    try:
        required = [cfg.sample_col, cfg.status_col, cfg.celltype_col, cfg.subtype_col]
        missing = [col for col in required if col not in adata.obs.columns]
        if missing:
            raise KeyError(f"Missing required obs columns: {missing}")
        obs = adata.obs[required].copy()
    finally:
        adata.file.close()
    return obs


def build_nonepi_frequency(obs: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    required = [cfg.sample_col, cfg.status_col, cfg.celltype_col, cfg.subtype_col]
    clean = obs.dropna(subset=required).copy()

    nonepi = clean.loc[clean[cfg.celltype_col].astype(str) != cfg.epi_label].copy()
    if nonepi.empty:
        raise ValueError("No nonepi cells remain after filtering cell_type != Epi.")

    nonepi_counts = nonepi.groupby(cfg.sample_col, observed=True).size().rename("nonepi_cells")
    keep_samples = nonepi_counts.index[nonepi_counts >= cfg.min_nonepi_cells_per_sample]
    nonepi = nonepi.loc[nonepi[cfg.sample_col].isin(keep_samples)].copy()
    if nonepi.empty:
        raise ValueError("No samples pass the minimum nonepi cell count threshold.")

    subtype_counts = pd.crosstab(nonepi[cfg.sample_col], nonepi[cfg.subtype_col]).astype(float)
    subtype_counts = subtype_counts.sort_index(axis=0).sort_index(axis=1)
    freq = subtype_counts.div(subtype_counts.sum(axis=1), axis=0).fillna(0.0)

    status_counts = pd.crosstab(nonepi[cfg.sample_col], nonepi[cfg.status_col]).reindex(freq.index).fillna(0)
    sample_status = pd.DataFrame(index=freq.index)
    sample_status["status"] = status_counts.idxmax(axis=1)
    sample_status["status_majority_fraction"] = status_counts.max(axis=1) / status_counts.sum(axis=1)
    for col in status_counts.columns:
        sample_status[f"n_{col}"] = status_counts[col].astype(int)
    sample_status["nonepi_cells"] = subtype_counts.sum(axis=1).astype(int)

    return freq, sample_status, subtype_counts



def build_epi_frequency(obs: pd.DataFrame, sample_index: pd.Index, cfg: Config) -> pd.DataFrame:
    required = [cfg.sample_col, cfg.celltype_col, cfg.subtype_col]
    clean = obs.dropna(subset=required).copy()
    epi = clean.loc[clean[cfg.celltype_col].astype(str).eq(cfg.epi_label)].copy()
    if epi.empty:
        return pd.DataFrame(index=sample_index)
    subtype_counts = pd.crosstab(epi[cfg.sample_col], epi[cfg.subtype_col]).astype(float)
    subtype_counts = subtype_counts.reindex(sample_index).fillna(0.0)
    subtype_counts = subtype_counts.sort_index(axis=0).sort_index(axis=1)
    epi_freq = subtype_counts.div(subtype_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    return epi_freq


def minmax_normalize_global(freq: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    mins = freq.min(axis=0)
    ranges = (freq.max(axis=0) - mins).replace(0, np.nan)
    normalized = ((freq - mins) / ranges).fillna(0.0).clip(lower=0.0)
    params = pd.DataFrame({"min": mins, "range": ranges.fillna(0.0)})
    return normalized, params


def make_group_balanced_weights(sample_status: pd.DataFrame, cfg: Config) -> pd.Series:
    status = sample_status["status"]
    normal_mask = status.eq(cfg.normal_status)
    tumor_mask = status.eq(cfg.tumor_status)
    if normal_mask.sum() == 0 or tumor_mask.sum() == 0:
        raise ValueError("Need both normal-like and tumor samples for group-balanced joint NMF.")

    weights = pd.Series(0.0, index=sample_status.index, name="joint_nmf_row_weight")
    weights.loc[normal_mask] = cfg.normal_group_total_weight / normal_mask.sum()
    weights.loc[tumor_mask] = cfg.tumor_group_total_weight / tumor_mask.sum()

    # Rescale to mean 1. This does not change relative group balance, but keeps
    # matrix scale comparable to unweighted NMF.
    weights = weights / weights.mean()
    return weights


# -----------------------------------------------------------------------------
# Rank selection and module classification
# -----------------------------------------------------------------------------


def evaluate_joint_rank(x_weighted: np.ndarray, cfg: Config) -> tuple[pd.DataFrame, int]:
    max_rank = min(cfg.module_k_range[1], x_weighted.shape[0], x_weighted.shape[1])
    min_rank = max(2, cfg.module_k_range[0])
    if max_rank < min_rank:
        raise ValueError(f"Invalid joint rank range after matrix size check: {min_rank}..{max_rank}")

    denom = float(np.square(x_weighted).sum())
    records: list[dict] = []

    for k in range(min_rank, max_rank + 1):
        seed_components: list[np.ndarray] = []
        seed_errors: list[float] = []
        seed_explained: list[float] = []

        for seed in cfg.rank_selection_seeds:
            model = make_nmf(n_components=k, random_state=seed, cfg=cfg)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UserWarning)
                w = model.fit_transform(x_weighted)
            h = model.components_
            recon = w @ h
            err = float(np.square(x_weighted - recon).sum())
            seed_components.append(h)
            seed_errors.append(err)
            seed_explained.append(1.0 - err / denom if denom else float("nan"))

        best_seed_i = int(np.argmin(seed_errors))
        records.append(
            {
                "k": k,
                "mean_balanced_explained_fraction": float(np.nanmean(seed_explained)),
                "best_balanced_explained_fraction": float(np.nanmax(seed_explained)),
                "mean_reconstruction_error": float(np.mean(seed_errors)),
                "best_reconstruction_error": float(np.min(seed_errors)),
                "stability_matched_cosine": stability_score(seed_components),
                "best_seed": int(cfg.rank_selection_seeds[best_seed_i]),
            }
        )

    metrics = pd.DataFrame(records)
    metrics["selection_score"] = (
        metrics["best_balanced_explained_fraction"].fillna(0.0)
        + 0.05 * metrics["stability_matched_cosine"].fillna(0.0)
        - 0.01 * metrics["k"]
    )

    if cfg.module_k is not None:
        selected_k = cfg.module_k
        if selected_k not in metrics["k"].tolist():
            raise ValueError(f"Forced module_k={selected_k} is outside evaluated ranks: {metrics['k'].tolist()}")
    else:
        selected_k = int(metrics.sort_values(["selection_score", "best_balanced_explained_fraction"], ascending=False).iloc[0]["k"])

    metrics["selected"] = metrics["k"].eq(selected_k)
    return metrics, selected_k


def classify_joint_modules(usage_all: pd.DataFrame, module_cols: list[str], cfg: Config) -> pd.DataFrame:
    normal = usage_all.loc[usage_all["status"].eq(cfg.normal_status), module_cols]
    tumor = usage_all.loc[usage_all["status"].eq(cfg.tumor_status), module_cols]
    if normal.empty or tumor.empty:
        raise ValueError("Need both normal-like and tumor samples to classify joint modules.")

    rows: list[dict] = []
    for module in module_cols:
        normal_mean = float(normal[module].mean())
        tumor_mean = float(tumor[module].mean())
        normal_median = float(normal[module].median())
        tumor_median = float(tumor[module].median())
        normal_active_fraction = float((normal[module] > 1e-8).mean())
        tumor_active_fraction = float((tumor[module] > 1e-8).mean())
        ratio = tumor_mean / normal_mean if normal_mean > 0 else np.inf

        if module in cfg.force_shared_modules:
            module_class = "sharedCM"
        elif module in cfg.force_tcm_modules:
            module_class = "tCM"
        elif module in cfg.force_normalcm_modules:
            module_class = "normalCM"
        elif ratio >= cfg.tumor_specific_min_ratio and tumor_active_fraction >= cfg.min_active_fraction_for_specific:
            module_class = "tCM"
        elif ratio <= cfg.normal_specific_max_ratio and normal_active_fraction >= cfg.min_active_fraction_for_specific:
            module_class = "normalCM"
        else:
            module_class = "sharedCM"

        display_module = f"{module}_{module_class}"
        rows.append(
            {
                "module": module,
                "display_module": display_module,
                "module_class": module_class,
                "normal_mean_usage": normal_mean,
                "tumor_mean_usage": tumor_mean,
                "tumor_to_normal_mean_ratio": ratio,
                "normal_median_usage": normal_median,
                "tumor_median_usage": tumor_median,
                "normal_active_fraction": normal_active_fraction,
                "tumor_active_fraction": tumor_active_fraction,
                "classification_rule": (
                    f"normalCM if ratio <= {cfg.normal_specific_max_ratio}; "
                    f"tCM if ratio >= {cfg.tumor_specific_min_ratio}; otherwise sharedCM"
                ),
            }
        )
    return pd.DataFrame(rows)



def benjamini_hochberg(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    adjusted = np.full_like(pvals, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    if valid.sum() == 0:
        return adjusted

    valid_p = pvals[valid]
    order = np.argsort(valid_p)
    ranked = valid_p[order]
    n = len(ranked)
    adj_ranked = ranked * n / np.arange(1, n + 1)
    adj_ranked = np.minimum.accumulate(adj_ranked[::-1])[::-1]
    adj_ranked = np.clip(adj_ranked, 0, 1)

    valid_indices = np.where(valid)[0]
    adjusted[valid_indices[order]] = adj_ranked
    return adjusted


def summarize_activity_by_status(activity_df: pd.DataFrame, sample_status: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    rows: list[dict] = []
    normal_samples = sample_status.index[sample_status["status"].eq(cfg.normal_status)]
    tumor_samples = sample_status.index[sample_status["status"].eq(cfg.tumor_status)]
    for cm in activity_df.columns:
        normal = activity_df.loc[normal_samples, cm].dropna()
        tumor = activity_df.loc[tumor_samples, cm].dropna()
        if len(normal) >= 2 and len(tumor) >= 2:
            stat, p_value = mannwhitneyu(tumor, normal, alternative="two-sided", method="auto")
            stat = float(stat)
            p_value = float(p_value)
        else:
            stat, p_value = np.nan, np.nan
        rows.append(
            {
                "CM": cm,
                f"{cfg.normal_status}_n": int(len(normal)),
                f"{cfg.normal_status}_mean": float(normal.mean()) if len(normal) else np.nan,
                f"{cfg.normal_status}_sd": float(normal.std(ddof=1)) if len(normal) > 1 else np.nan,
                f"{cfg.normal_status}_median": float(normal.median()) if len(normal) else np.nan,
                f"{cfg.tumor_status}_n": int(len(tumor)),
                f"{cfg.tumor_status}_mean": float(tumor.mean()) if len(tumor) else np.nan,
                f"{cfg.tumor_status}_sd": float(tumor.std(ddof=1)) if len(tumor) > 1 else np.nan,
                f"{cfg.tumor_status}_median": float(tumor.median()) if len(tumor) else np.nan,
                "mean_diff_tumor_minus_normal": (float(tumor.mean()) - float(normal.mean())) if len(tumor) and len(normal) else np.nan,
                "median_diff_tumor_minus_normal": (float(tumor.median()) - float(normal.median())) if len(tumor) and len(normal) else np.nan,
                "MWU_U_tumor_vs_normal": stat,
                "MWU_p_tumor_vs_normal": p_value,
                "direction_by_mean": "tumor_high" if len(tumor) and len(normal) and tumor.mean() > normal.mean() else "normal_high" if len(tumor) and len(normal) and tumor.mean() < normal.mean() else "equal",
            }
        )
    summary = pd.DataFrame(rows)
    ok = summary["MWU_p_tumor_vs_normal"].notna()
    summary["MWU_q_tumor_vs_normal_BH"] = np.nan
    if ok.any():
        summary.loc[ok, "MWU_q_tumor_vs_normal_BH"] = benjamini_hochberg(summary.loc[ok, "MWU_p_tumor_vs_normal"].to_numpy(dtype=float))
    summary["significant_q_lt_0.05"] = summary["MWU_q_tumor_vs_normal_BH"] < 0.05
    return summary


def add_module_class_to_activity_summary(summary_df: pd.DataFrame, classification: pd.DataFrame) -> pd.DataFrame:
    class_map = classification.set_index("display_module")["module_class"].to_dict()
    original_map = classification.set_index("display_module")["module"].to_dict()
    out = summary_df.copy()
    out.insert(1, "original_module", out["CM"].map(original_map))
    out.insert(2, "module_class", out["CM"].map(class_map))
    return out


def epi_cm_association(
    epi_freq_df: pd.DataFrame,
    activity_df: pd.DataFrame,
    sample_status: pd.DataFrame,
    cfg: Config,
    subset_group: str,
    reference_group: str = "joint_cm",
    reference_prefix: str = "balanced_joint_cm",
    method: str = "spearman",
) -> pd.DataFrame:
    samples = sample_status.index[sample_status["status"].eq(subset_group)]
    samples = samples.intersection(epi_freq_df.index).intersection(activity_df.index)
    rows: list[dict] = []
    for epi in epi_freq_df.columns:
        x = epi_freq_df.loc[samples, epi]
        for cm_name in activity_df.columns:
            y = activity_df.loc[samples, cm_name]
            if len(samples) < 4 or x.std(ddof=0) == 0 or y.std(ddof=0) == 0:
                rho, p = np.nan, np.nan
            elif method == "spearman":
                rho, p = spearmanr(x, y)
            elif method == "pearson":
                rho, p = pearsonr(x, y)
            else:
                raise ValueError(f"Unsupported correlation method: {method}")
            rows.append(
                {
                    "reference_group": reference_group,
                    "reference_prefix": reference_prefix,
                    "group": subset_group,
                    "epi_subtype": epi,
                    "CM": cm_name,
                    "rho": float(rho) if pd.notna(rho) else np.nan,
                    "p": float(p) if pd.notna(p) else np.nan,
                    "n_samples": int(len(samples)),
                    "method": method,
                }
            )
    res = pd.DataFrame(rows)
    res["q"] = np.nan
    ok = res["p"].notna()
    if ok.any():
        res.loc[ok, "q"] = benjamini_hochberg(res.loc[ok, "p"].to_numpy(dtype=float))
    return res.sort_values(["q", "p"], na_position="last")


def association_matrix(assoc_df: pd.DataFrame, value: str) -> pd.DataFrame:
    return assoc_df.pivot(index="epi_subtype", columns="CM", values=value)


def save_association_matrices(prefix: str, group: str, assoc_df: pd.DataFrame, output_dir: Path) -> None:
    association_matrix(assoc_df, "rho").to_csv(output_dir / f"{prefix}_epi_cm_association_{group}_rho_matrix.csv")
    association_matrix(assoc_df, "q").to_csv(output_dir / f"{prefix}_epi_cm_association_{group}_q_matrix.csv")




def fisher_corr_diff_p(normal_rho: float, normal_n: int, tumor_rho: float, tumor_n: int) -> float:
    if any(pd.isna(v) for v in [normal_rho, normal_n, tumor_rho, tumor_n]) or normal_n <= 3 or tumor_n <= 3:
        return np.nan
    normal_rho = float(np.clip(normal_rho, -0.999999, 0.999999))
    tumor_rho = float(np.clip(tumor_rho, -0.999999, 0.999999))
    z = (np.arctanh(tumor_rho) - np.arctanh(normal_rho)) / np.sqrt(1 / (tumor_n - 3) + 1 / (normal_n - 3))
    return float(2 * norm.sf(abs(z)))


def epi_cm_coupling_rewiring(
    normal_assoc: pd.DataFrame,
    tumor_assoc: pd.DataFrame,
    reference_group: str = "joint_cm",
    reference_prefix: str = "balanced_joint_cm",
) -> pd.DataFrame:
    normal = normal_assoc[["epi_subtype", "CM", "rho", "p", "q", "n_samples"]].rename(
        columns={"rho": "normal_rho", "p": "normal_p", "q": "normal_q", "n_samples": "normal_n_samples"}
    )
    tumor = tumor_assoc[["epi_subtype", "CM", "rho", "p", "q", "n_samples"]].rename(
        columns={"rho": "tumor_rho", "p": "tumor_p", "q": "tumor_q", "n_samples": "tumor_n_samples"}
    )
    out = normal.merge(tumor, on=["epi_subtype", "CM"], how="inner")
    out["delta_rho_tumor_minus_normal_like"] = out["tumor_rho"] - out["normal_rho"]
    out["p_fisher_delta"] = out.apply(
        lambda row: fisher_corr_diff_p(row["normal_rho"], row["normal_n_samples"], row["tumor_rho"], row["tumor_n_samples"]),
        axis=1,
    )
    out["q_fisher_delta"] = np.nan
    ok = out["p_fisher_delta"].notna()
    if ok.any():
        out.loc[ok, "q_fisher_delta"] = benjamini_hochberg(out.loc[ok, "p_fisher_delta"].to_numpy(dtype=float))
    out.insert(0, "reference_group", reference_group)
    out.insert(1, "reference_prefix", reference_prefix)
    return out.sort_values("delta_rho_tumor_minus_normal_like", key=lambda x: x.abs(), ascending=False)




def write_epi_cm_coupling_outputs(
    epi_freq_df: pd.DataFrame,
    activity_df: pd.DataFrame,
    sample_status: pd.DataFrame,
    cfg: Config,
    output_dir: Path,
) -> dict[str, pd.DataFrame]:
    reference_group = "joint_cm"
    reference_prefix = "balanced_joint_cm"
    association_results: dict[str, pd.DataFrame] = {}
    for group in [cfg.normal_status, cfg.tumor_status]:
        assoc = epi_cm_association(
            epi_freq_df,
            activity_df,
            sample_status,
            cfg,
            subset_group=group,
            reference_group=reference_group,
            reference_prefix=reference_prefix,
            method="spearman",
        )
        association_results[group] = assoc
        assoc.to_csv(output_dir / f"{reference_prefix}_epi_cm_association_{group}.csv", index=False)
        save_association_matrices(reference_prefix, group, assoc, output_dir)

    coupling = epi_cm_coupling_rewiring(
        association_results[cfg.normal_status],
        association_results[cfg.tumor_status],
        reference_group=reference_group,
        reference_prefix=reference_prefix,
    )
    coupling.to_csv(output_dir / f"{reference_prefix}_epi_cm_coupling_rewiring_tumor_minus_normal_like.csv", index=False)
    coupling.pivot(index="epi_subtype", columns="CM", values="delta_rho_tumor_minus_normal_like").to_csv(
        output_dir / f"{reference_prefix}_epi_cm_coupling_rewiring_tumor_minus_normal_like_delta_rho_matrix.csv"
    )
    write_hcm_to_joint_cm_coupling_file_mapping(output_dir, reference_prefix)
    return {**association_results, "coupling_rewiring": coupling}


def write_hcm_to_joint_cm_coupling_file_mapping(output_dir: Path, reference_prefix: str) -> None:
    expected_pairs = [
        ("normal_reference_hcm_epi_cm_association_normal-like.csv", f"{reference_prefix}_epi_cm_association_normal-like.csv"),
        ("normal_reference_hcm_epi_cm_association_tumor.csv", f"{reference_prefix}_epi_cm_association_tumor.csv"),
        ("normal_reference_hcm_epi_cm_coupling_rewiring_tumor_minus_normal_like.csv", f"{reference_prefix}_epi_cm_coupling_rewiring_tumor_minus_normal_like.csv"),
        ("normal_reference_hcm_epi_cm_association_normal-like_q_matrix.csv", f"{reference_prefix}_epi_cm_association_normal-like_q_matrix.csv"),
        ("normal_reference_hcm_epi_cm_association_normal-like_rho_matrix.csv", f"{reference_prefix}_epi_cm_association_normal-like_rho_matrix.csv"),
        ("normal_reference_hcm_epi_cm_association_tumor_q_matrix.csv", f"{reference_prefix}_epi_cm_association_tumor_q_matrix.csv"),
        ("normal_reference_hcm_epi_cm_association_tumor_rho_matrix.csv", f"{reference_prefix}_epi_cm_association_tumor_rho_matrix.csv"),
        ("normal_reference_hcm_epi_cm_coupling_rewiring_tumor_minus_normal_like_delta_rho_matrix.csv", f"{reference_prefix}_epi_cm_coupling_rewiring_tumor_minus_normal_like_delta_rho_matrix.csv"),
    ]
    rows = []
    for hcm_file, joint_file in expected_pairs:
        path = output_dir / joint_file
        rows.append(
            {
                "hcm_expected_file": hcm_file,
                "joint_cm_corresponding_file": joint_file,
                "exists": path.exists(),
                "size_bytes": path.stat().st_size if path.exists() else 0,
            }
        )
    pd.DataFrame(rows).to_csv(output_dir / "hcm_to_joint_cm_epi_cm_coupling_file_mapping.csv", index=False)


# -----------------------------------------------------------------------------
# Outputs
# -----------------------------------------------------------------------------


def component_frame(h: np.ndarray, feature_names: list[str], prefix: str = "joint") -> pd.DataFrame:
    names = [f"{prefix}_{i + 1:02d}" for i in range(h.shape[0])]
    return pd.DataFrame(h, index=names, columns=feature_names)


def usage_frame(w: np.ndarray, sample_names: pd.Index, module_names: list[str]) -> pd.DataFrame:
    return pd.DataFrame(w, index=sample_names, columns=module_names)


def display_module_map(classification: pd.DataFrame) -> dict[str, str]:
    return classification.set_index("module")["display_module"].to_dict()


def relabel_module_index(df: pd.DataFrame, classification: pd.DataFrame) -> pd.DataFrame:
    return df.rename(index=display_module_map(classification))


def relabel_module_columns(df: pd.DataFrame, classification: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns=display_module_map(classification))


def split_and_save_by_class(h_df: pd.DataFrame, usage_all: pd.DataFrame, classification: pd.DataFrame, out: dict[str, Path]) -> None:
    name_map = display_module_map(classification)
    for module_class in ["sharedCM", "tCM", "normalCM"]:
        modules = classification.loc[classification["module_class"].eq(module_class), "module"].tolist()
        h_part = h_df.loc[modules].rename(index=name_map) if modules else pd.DataFrame(columns=h_df.columns)
        h_part_frac = h_part.div(h_part.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
        usage_cols = modules + ["status", "nonepi_cells"]
        usage_part = usage_all[usage_cols].rename(columns=name_map) if modules else usage_all[["status", "nonepi_cells"]].copy()

        stem = module_class.lower()
        h_part.to_csv(out["module_groups"] / f"{stem}_subtype_loadings_raw.csv")
        h_part_frac.to_csv(out["module_groups"] / f"{stem}_subtype_loadings_fraction.csv")
        usage_part.to_csv(out["module_groups"] / f"{stem}_usage_all_samples_nnls.csv")


def top_subtype_table(h_df: pd.DataFrame, classification: pd.DataFrame, top_n: int) -> pd.DataFrame:
    class_map = classification.set_index("module")["module_class"].to_dict()
    display_map = display_module_map(classification)
    h_frac = h_df.div(h_df.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    rows: list[dict] = []
    for module in h_df.index:
        ordered = h_df.loc[module].sort_values(ascending=False).head(top_n)
        for rank, subtype in enumerate(ordered.index, start=1):
            rows.append(
                {
                    "module_class": class_map.get(module, "unclassified"),
                    "module": display_map.get(module, module),
                    "original_module": module,
                    "rank": rank,
                    "cell_subtype": subtype,
                    "loading_raw": float(h_df.loc[module, subtype]),
                    "loading_fraction": float(h_frac.loc[module, subtype]),
                }
            )
    return pd.DataFrame(rows)





























def get_top_nodes_from_loading_df(loading_df: pd.DataFrame, top_n: int = 10) -> dict[str, list[str]]:
    return {
        cm_name: loading_df[cm_name].sort_values(ascending=False).head(top_n).index.tolist()
        for cm_name in loading_df.columns
    }


def cm_correlation_matrices_for_subset(
    freq_df: pd.DataFrame,
    loading_df: pd.DataFrame,
    samples: pd.Index,
    top_n: int = 10,
    method: str = "pearson",
    node_sets: dict[str, list[str]] | None = None,
) -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    if node_sets is None:
        node_sets = get_top_nodes_from_loading_df(loading_df, top_n)
    samples = pd.Index(samples).intersection(freq_df.index)
    corr_matrices: dict[str, pd.DataFrame] = {}
    node_weight_tables: dict[str, pd.DataFrame] = {}
    for cm_name, raw_nodes in node_sets.items():
        nodes = [node for node in raw_nodes if node in freq_df.columns and node in loading_df.index]
        node_weights = loading_df.loc[nodes, cm_name].sort_values(ascending=False)
        ordered_nodes = node_weights.index.tolist()
        node_weight_tables[cm_name] = pd.DataFrame(
            {
                "CM": cm_name,
                "node": ordered_nodes,
                "weight": node_weights.values,
                "rank": np.arange(1, len(node_weights) + 1),
            }
        )
        if len(ordered_nodes) < 2 or len(samples) < 4:
            corr_matrices[cm_name] = pd.DataFrame(index=ordered_nodes, columns=ordered_nodes, dtype=float)
            continue
        corr_matrices[cm_name] = freq_df.loc[samples, ordered_nodes].corr(method=method).loc[ordered_nodes, ordered_nodes]
    return corr_matrices, node_weight_tables


def reference_node_sets_from_edges(
    corr_matrices: dict[str, pd.DataFrame],
    node_weight_tables: dict[str, pd.DataFrame],
    threshold: float,
) -> dict[str, list[str]]:
    reference_node_sets: dict[str, list[str]] = {}
    for cm_name, corr_df in corr_matrices.items():
        ranked_nodes = node_weight_tables[cm_name]["node"].tolist()
        connected_nodes: set[str] = set()
        for i, node_a in enumerate(ranked_nodes):
            for node_b in ranked_nodes[i + 1:]:
                value = corr_df.loc[node_a, node_b] if node_a in corr_df.index and node_b in corr_df.columns else np.nan
                if pd.notna(value) and float(value) >= threshold:
                    connected_nodes.update([node_a, node_b])
        reference_node_sets[cm_name] = [node for node in ranked_nodes if node in connected_nodes]
    return reference_node_sets










def save_node_weight_table(prefix: str, node_weight_tables: dict[str, pd.DataFrame], output_dir: Path) -> pd.DataFrame:
    if node_weight_tables:
        node_weight_df = pd.concat(node_weight_tables.values(), ignore_index=True)
    else:
        node_weight_df = pd.DataFrame(columns=["CM", "node", "weight", "rank"])
    node_weight_df.to_csv(output_dir / f"{prefix}_network_nodes_from_loading_df.csv", index=False)
    return node_weight_df


def save_cm_node_alias_table(node_weight_df: pd.DataFrame, output_dir: Path, filename: str) -> None:
    node_weight_df.to_csv(output_dir / filename, index=False)


def build_cm_cell_subtype_node_table(
    h_df: pd.DataFrame,
    loading_df_fraction: pd.DataFrame,
    classification: pd.DataFrame,
) -> pd.DataFrame:
    class_map = classification.set_index("display_module")["module_class"].to_dict()
    original_map = classification.set_index("display_module")["module"].to_dict()
    rows: list[dict] = []
    for cm_name in h_df.index:
        raw = h_df.loc[cm_name].astype(float).sort_values(ascending=False)
        frac = loading_df_fraction[cm_name].astype(float).reindex(raw.index)
        for rank, cell_subtype in enumerate(raw.index, start=1):
            rows.append(
                {
                    "module_class": class_map.get(cm_name),
                    "CM": cm_name,
                    "original_module": original_map.get(cm_name),
                    "rank": rank,
                    "cell_subtype": cell_subtype,
                    "node": cell_subtype,
                    "loading_raw": float(raw.loc[cell_subtype]),
                    "loading_fraction": float(frac.loc[cell_subtype]) if pd.notna(frac.loc[cell_subtype]) else np.nan,
                }
            )
    return pd.DataFrame(rows)


def save_cm_cell_subtype_node_outputs(
    h_df: pd.DataFrame,
    loading_df_fraction: pd.DataFrame,
    classification: pd.DataFrame,
    out: dict[str, Path],
    cfg: Config,
) -> pd.DataFrame:
    # Current joint NMF uses X(sample x cell_subtype) ~= W(sample x CM) @ H(CM x cell_subtype).
    # Therefore CM node/cell_subtype tables are derived from H_df, not W_df.
    node_table = build_cm_cell_subtype_node_table(h_df, loading_df_fraction, classification)
    node_table.to_csv(out["modules"] / "joint_cm_cell_subtype_nodes_all_from_H_df.csv", index=False)
    node_table.loc[node_table["rank"] <= cfg.top_n_nodes].to_csv(
        out["modules"] / f"joint_cm_cell_subtype_nodes_top{cfg.top_n_nodes}_from_H_df.csv",
        index=False,
    )
    node_table.loc[node_table["rank"] <= cfg.top_n_subtypes].to_csv(
        out["modules"] / f"joint_cm_cell_subtype_nodes_top{cfg.top_n_subtypes}_from_H_df.csv",
        index=False,
    )
    h_df.to_csv(out["modules"] / "balanced_joint_cm_subtype_loadings_raw_from_H_df.csv")
    loading_df_fraction.T.to_csv(out["modules"] / "balanced_joint_cm_subtype_loadings_fraction_from_H_df.csv")
    return node_table


def save_reference_node_sets(prefix: str, reference_node_sets: dict[str, list[str]], output_dir: Path) -> pd.DataFrame:
    rows: list[dict] = []
    for cm_name, nodes in reference_node_sets.items():
        for rank, node in enumerate(nodes, start=1):
            rows.append({"CM": cm_name, "reference_node_rank": rank, "node": node})
    ref_nodes_df = pd.DataFrame(rows)
    ref_nodes_df.to_csv(output_dir / f"{prefix}_reference_node_sets_after_edge_threshold.csv", index=False)
    return ref_nodes_df






# -----------------------------------------------------------------------------
# Main workflow
# -----------------------------------------------------------------------------



def run_balanced_joint_nmf_analysis() -> None:
    base_dir = Path(CFG.project_dir).resolve()
    input_h5ad = base_dir / CFG.input_h5ad
    output_base = base_dir / CFG.output_dir
    if output_base.exists():
        print(f"Clearing existing output directory: {output_base}")
        shutil.rmtree(output_base)
    out = ensure_dirs(output_base)

    if not input_h5ad.exists():
        raise FileNotFoundError(input_h5ad)

    write_json(asdict(CFG), out["modules"] / "run_config.json")

    print(f"Reading obs from: {input_h5ad}")
    obs = load_obs(input_h5ad, CFG)

    print("Building sample x nonepi subtype fraction matrix...")
    freq, sample_status, subtype_counts = build_nonepi_frequency(obs, CFG)
    freq.to_csv(out["shared"] / "non_epi_subtype_frequency.csv")
    subtype_counts.to_csv(out["shared"] / "non_epi_subtype_counts.csv")
    sample_status.to_csv(out["shared"] / "sample_status.csv")

    epi_freq = build_epi_frequency(obs, freq.index, CFG)
    epi_freq.to_csv(out["shared"] / "epi_subtype_frequency.csv")

    normal_samples = sample_status.index[sample_status["status"].eq(CFG.normal_status)]
    tumor_samples = sample_status.index[sample_status["status"].eq(CFG.tumor_status)]
    print(f"Normal-like samples: {len(normal_samples)}")
    print(f"Tumor samples: {len(tumor_samples)}")

    norm_df, norm_params = minmax_normalize_global(freq)
    norm_df.to_csv(out["shared"] / "non_epi_subtype_frequency_global_minmax.csv")
    norm_params.to_csv(out["shared"] / "global_minmax_params.csv")

    row_weights = make_group_balanced_weights(sample_status, CFG)
    row_weights.to_frame().join(sample_status[["status", "nonepi_cells"]]).to_csv(out["shared"] / "group_balanced_sample_weights.csv")

    x = norm_df.to_numpy(dtype=float)
    x_weighted = x * np.sqrt(row_weights.loc[norm_df.index].to_numpy(dtype=float))[:, None]

    print("Selecting total joint NMF rank on group-balanced matrix...")
    rank_metrics, selected_k = evaluate_joint_rank(x_weighted, CFG)
    rank_metrics.to_csv(out["rank_selection"] / "joint_nmf_k_selection_metrics.csv", index=False)
    write_json({"selected_module_k": selected_k}, out["rank_selection"] / "selected_module_k.json")
    print(f"Selected total joint module k: {selected_k}")

    print("Fitting final group-balanced joint NMF...")
    final_seed = int(rank_metrics.loc[rank_metrics["k"].eq(selected_k), "best_seed"].iloc[0])
    _, w_weighted, h_joint = fit_nmf_best_seed(
        x_weighted,
        n_components=selected_k,
        seeds=(final_seed,),
        cfg=CFG,
    )

    feature_names = norm_df.columns.tolist()
    h_df = component_frame(h_joint, feature_names, "joint")

    print("Refitting all samples to fixed joint basis by NNLS on the unweighted matrix...")
    w_all = fit_nnls_usage(h_joint, x)
    module_names = h_df.index.tolist()
    usage_all = usage_frame(w_all, norm_df.index, module_names)
    usage_all = usage_all.join(sample_status[["status", "nonepi_cells"]])

    print("Classifying joint modules into sharedCM, tCM, and normalCM...")
    classification = classify_joint_modules(usage_all, module_names, CFG)
    classification.to_csv(out["module_groups"] / "joint_module_classification.csv", index=False)

    h_df = relabel_module_index(h_df, classification)  # NMF H: CM x cell_subtype
    w_df = relabel_module_columns(usage_all[module_names], classification)  # NMF W / NNLS usage: sample x CM

    # File names follow the current NMF input direction:
    # X(sample x cell_subtype) ~= W(sample x CM) @ H(CM x cell_subtype).
    w_df.to_csv(out["modules"] / "W_df.csv")
    h_df.to_csv(out["modules"] / "H_df.csv")

    # Derived meaning-based matrices used by plots and downstream CM scoring.
    loading_df = h_df.T.copy()  # derived from H: cell_subtype x CM
    loading_df_fraction = loading_df.div(loading_df.sum(axis=0).replace(0, np.nan), axis=1).fillna(0.0)
    activity_cm_by_sample_df = w_df.T.copy()  # derived from W: CM x sample

    loading_df.to_csv(out["modules"] / "loading_df_cell_subtype_by_CM.csv")
    loading_df_fraction.to_csv(out["modules"] / "loading_df_cell_subtype_by_CM_fraction.csv")
    activity_cm_by_sample_df.to_csv(out["modules"] / "activity_df_CM_by_sample.csv")
    w_df.join(sample_status[["status", "nonepi_cells"]]).to_csv(out["modules"] / "activity_df_sample_by_CM.csv")

    # Precompute and save the retained heatmap matrices here. The plots notebook
    # only reads these CSV files and renders figures.
    loading_df.to_csv(out["modules"] / "h_df_loading_cell_subtype_by_CM_raw.csv")
    loading_min = loading_df.min(axis=0)
    loading_range = (loading_df.max(axis=0) - loading_min).replace(0, np.nan)
    loading_standard_scale_col = loading_df.sub(loading_min, axis=1).div(loading_range, axis=1).fillna(0.0).clip(0.0, 1.0)
    loading_standard_scale_col.to_csv(out["modules"] / "h_df_loading_cell_subtype_by_CM_standard_scale_col.csv")

    activity_sample_by_cm = activity_cm_by_sample_df.T.loc[
        sample_status.index.intersection(activity_cm_by_sample_df.columns)
    ].copy()
    activity_sample_by_cm.to_csv(out["modules"] / "w_df_activity_sample_by_CM_raw.csv")
    activity_min = activity_sample_by_cm.min(axis=0)
    activity_range = (activity_sample_by_cm.max(axis=0) - activity_min).replace(0, np.nan)
    activity_standard_scale_col = activity_sample_by_cm.sub(activity_min, axis=1).div(activity_range, axis=1).fillna(0.0).clip(0.0, 1.0)
    activity_standard_scale_col.to_csv(out["modules"] / "w_df_activity_sample_by_CM_standard_scale_col.csv")
    pd.DataFrame(
        [
            {
                "file": "W_df.csv",
                "source": "NMF W / NNLS-refit usage",
                "orientation": "sample x CM",
                "rows": "sample",
                "columns": "CM",
                "purpose": "sample-level CM activity; matches W in X ~= W @ H",
            },
            {
                "file": "H_df.csv",
                "source": "NMF H learned basis",
                "orientation": "CM x cell_subtype",
                "rows": "CM",
                "columns": "cell_subtype",
                "purpose": "CM subtype loading; matches H in X ~= W @ H",
            },
            {
                "file": "loading_df_cell_subtype_by_CM.csv",
                "source": "transpose of H_df.csv",
                "orientation": "cell_subtype x CM",
                "rows": "cell_subtype",
                "columns": "CM",
                "purpose": "loading heatmap, top subtype tables, network nodes, downstream CM scoring",
            },
            {
                "file": "loading_df_cell_subtype_by_CM_fraction.csv",
                "source": "column-normalized loading_df_cell_subtype_by_CM.csv",
                "orientation": "cell_subtype x CM",
                "rows": "cell_subtype",
                "columns": "CM",
                "purpose": "fractional subtype contribution per CM",
            },
            {
                "file": "activity_df_CM_by_sample.csv",
                "source": "transpose of W_df.csv",
                "orientation": "CM x sample",
                "rows": "CM",
                "columns": "sample",
                "purpose": "activity heatmap variants and CM x sample activity layout",
            },
            {
                "file": "activity_df_sample_by_CM.csv",
                "source": "W_df.csv with sample annotations joined",
                "orientation": "sample x CM plus status/nonepi_cells",
                "rows": "sample",
                "columns": "CM plus metadata",
                "purpose": "sample-level activity table for statistics and MWU grouping",
            },
            {
                "file": "h_df_loading_cell_subtype_by_CM_{raw,standard_scale_col}.csv",
                "source": "column-wise transforms of loading_df_cell_subtype_by_CM.csv, derived from H_df.csv",
                "orientation": "cell_subtype x CM",
                "rows": "cell_subtype",
                "columns": "CM",
                "purpose": "H_df-derived loading heatmap tables; transforms are per CM column",
            },
            {
                "file": "w_df_activity_sample_by_CM_{raw,standard_scale_col}.csv",
                "source": "column-wise transforms of W_df.csv before saving as CM x sample",
                "orientation": "sample x CM",
                "rows": "sample",
                "columns": "CM",
                "purpose": "W_df-derived activity heatmap tables; same orientation as plotted heatmap, transforms are per CM column",
            },
        ]
    ).to_csv(out["modules"] / "matrix_orientation_readme.csv", index=False)

    split_and_save_by_class(h_df=h_df.rename(index={v:k for k,v in display_module_map(classification).items()}), usage_all=usage_all, classification=classification, out=out)

    print("Writing H_df-derived CM cell_subtype/node tables...")
    save_cm_cell_subtype_node_outputs(h_df, loading_df_fraction, classification, out, CFG)

    print("Writing epi-CM coupling outputs...")
    write_epi_cm_coupling_outputs(epi_freq, w_df, sample_status, CFG, out["epi_cm_coupling"])

    print("Writing CM activity summary and MWU...")
    activity_summary_df = summarize_activity_by_status(w_df, sample_status, CFG)
    activity_summary_df = add_module_class_to_activity_summary(activity_summary_df, classification)
    activity_summary_df.to_csv(out["stats"] / "joint_CM_activity_tumor_vs_normal_mean_sd_summary.csv", index=False)
    activity_summary_df.to_csv(out["mwu"] / "CM_activity_tumor_vs_normal_MWU.csv", index=False)

    print("Computing status-specific CM node tables from loading top nodes...")
    normal_samples_for_network = sample_status.index[sample_status["status"].eq(CFG.normal_status)]
    tumor_samples_for_network = sample_status.index[sample_status["status"].eq(CFG.tumor_status)]
    normal_top_corr_matrices, normal_top_node_tables = cm_correlation_matrices_for_subset(norm_df, loading_df, normal_samples_for_network, top_n=CFG.top_n_nodes)
    tumor_top_corr_matrices, tumor_top_node_tables = cm_correlation_matrices_for_subset(norm_df, loading_df, tumor_samples_for_network, top_n=CFG.top_n_nodes)
    normal_node_sets = reference_node_sets_from_edges(normal_top_corr_matrices, normal_top_node_tables, CFG.edge_r_threshold)
    tumor_node_sets = reference_node_sets_from_edges(tumor_top_corr_matrices, tumor_top_node_tables, CFG.edge_r_threshold)
    retained_node_sets = {
        cm_name: [node for node in get_top_nodes_from_loading_df(loading_df, CFG.top_n_nodes)[cm_name] if node in set(normal_node_sets.get(cm_name, [])).union(tumor_node_sets.get(cm_name, []))]
        for cm_name in loading_df.columns
    }
    reference_node_table = save_reference_node_sets("status_specific", retained_node_sets, out["networks"])
    normal_corr_matrices, normal_node_weight_tables = cm_correlation_matrices_for_subset(norm_df, loading_df, normal_samples_for_network, top_n=CFG.top_n_nodes, node_sets=retained_node_sets)
    tumor_corr_matrices, tumor_node_weight_tables = cm_correlation_matrices_for_subset(norm_df, loading_df, tumor_samples_for_network, top_n=CFG.top_n_nodes, node_sets=retained_node_sets)
    normal_node_table = save_node_weight_table("normal_like", normal_node_weight_tables, out["networks"])
    tumor_node_table = save_node_weight_table("tumor", tumor_node_weight_tables, out["networks"])
    reference_node_table.to_csv(out["networks"] / "balanced_joint_cm_reference_node_sets_after_edge_threshold.csv", index=False)
    save_cm_node_alias_table(normal_node_table, out["networks"], "normal_like_network_nodes_from_H_df.csv")
    save_cm_node_alias_table(tumor_node_table, out["networks"], "tumor_network_nodes_from_H_df.csv")
    save_cm_node_alias_table(normal_node_table, out["networks"], "balanced_joint_cm_reference_normal_like_network_nodes_from_H_df.csv")
    save_cm_node_alias_table(tumor_node_table, out["networks"], "balanced_joint_cm_to_tumor_network_nodes_from_H_df.csv")

    edge_rows = []
    for context, corr_matrices in [("normal-like", normal_corr_matrices), ("tumor", tumor_corr_matrices)]:
        for cm_name, corr_df in corr_matrices.items():
            nodes = corr_df.index.tolist()
            for i, node_a in enumerate(nodes):
                for node_b in nodes[i + 1:]:
                    value = corr_df.loc[node_a, node_b]
                    edge_rows.append({
                        "context": context,
                        "CM": cm_name,
                        "node_a": node_a,
                        "node_b": node_b,
                        "pearson_r": value,
                        "edge_pass_r_ge_0.25": bool(pd.notna(value) and float(value) >= CFG.edge_r_threshold),
                    })
    pd.DataFrame(edge_rows).to_csv(out["networks"] / "status_specific_nodeplot_edges.csv", index=False)


    recon_all = w_all @ h_joint
    reconstruction = pd.DataFrame(index=norm_df.index)
    reconstruction.index.name = "sample"
    reconstruction["balanced_joint_nmf_explained_fraction"] = explained_fraction_by_row(x, recon_all)
    reconstruction = reconstruction.join(sample_status[["status", "nonepi_cells"]])
    reconstruction.reset_index().to_csv(out["combined"] / "reconstruction_per_sample.csv", index=False)

    overall_summary = pd.DataFrame(
        [
            {
                "n_samples_total": int(freq.shape[0]),
                "n_normal_like_samples": int(len(normal_samples)),
                "n_tumor_samples": int(len(tumor_samples)),
                "n_nonepi_subtypes": int(freq.shape[1]),
                "selected_module_k": int(selected_k),
                "n_sharedcm": int((classification["module_class"] == "sharedCM").sum()),
                "n_tcm": int((classification["module_class"] == "tCM").sum()),
                "n_normalcm": int((classification["module_class"] == "normalCM").sum()),
                "overall_explained_fraction_unweighted": explained_fraction(x, recon_all),
                "overall_explained_fraction_group_balanced": explained_fraction(x_weighted, w_weighted @ h_joint),
                "normal_mean_explained_fraction": float(reconstruction.loc[normal_samples, "balanced_joint_nmf_explained_fraction"].mean()),
                "tumor_mean_explained_fraction": float(reconstruction.loc[tumor_samples, "balanced_joint_nmf_explained_fraction"].mean()),
            }
        ]
    )
    overall_summary.to_csv(out["combined"] / "reconstruction_overall_summary.csv", index=False)

    top_table = top_subtype_table(h_df.rename(index={v:k for k,v in display_module_map(classification).items()}), classification, CFG.top_n_subtypes)
    top_table.to_csv(out["summary"] / "top_subtypes_per_module.csv", index=False)


    print("Done.")
    print(f"Outputs written to: {out['base']}")


if __name__ == "__main__":
    run_balanced_joint_nmf_analysis()


## Output contract

This notebook performs the joint NMF analysis and saves all CSV/JSON tables required downstream. It does not draw or save figures. Run `balanced_joint_nmf_cm_decomposition_plots.ipynb` after this notebook to generate the retained figures.


# 2. Selected joint NMF heatmaps


# Selected Group-Balanced Joint NMF Figures

Run `balanced_joint_nmf_cm_decomposition.ipynb` first. This notebook only reads precomputed CSV tables and renders the retained figures.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


PROJECT_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_reseach/CM_analysis_2/cm_epi_analysis")
OUTPUT_DIR = PROJECT_DIR / "balanced_joint_nmf_outputs" / "joint_cm"
TABLE_DIR = OUTPUT_DIR / "tables"
EPI_CM_DIR = OUTPUT_DIR / "epi_cm_coupling"
FIGURE_DIR = OUTPUT_DIR / "figures"

EPI_CM_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "Arial"
sns.set_theme(style="white", font="Arial")


## Tumor epithelial-state–CM association heatmap


In [ ]:
association_matrix = pd.read_csv(
    EPI_CM_DIR / "balanced_joint_cm_epi_cm_association_tumor_rho_matrix.csv",
    index_col=0,
)

fig, ax = plt.subplots(
    figsize=(max(5, association_matrix.shape[1] * 0.55), max(4, association_matrix.shape[0] * 0.35))
)
sns.heatmap(
    association_matrix,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Spearman rho"},
    ax=ax,
)
ax.set_title("Balanced joint CM: epithelial states vs CMs in tumor")
fig.tight_layout()
fig.savefig(
    EPI_CM_DIR / "balanced_joint_cm_epi_cm_association_tumor_heatmap.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.close(fig)


## H_df loading clustermap (column min–max)


In [ ]:
loading_standard_scale_col = pd.read_csv(
    TABLE_DIR / "h_df_loading_cell_subtype_by_CM_standard_scale_col.csv",
    index_col=0,
)

g = sns.clustermap(
    loading_standard_scale_col,
    cmap="viridis",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Column min-max loading"},
    figsize=(loading_standard_scale_col.shape[1] * 0.35 + 2.5, loading_standard_scale_col.shape[0] * 0.13 + 2.5),
    linewidths=0,
    col_cluster=False,
)
g.fig.suptitle("H_df loading: cell subtype weights per CM (column min-max)", y=1.02)
for spine in g.ax_heatmap.spines.values():
    spine.set_visible(False)
g.fig.tight_layout(rect=[0, 0, 1, 0.98])
g.savefig(
    FIGURE_DIR / "h_df_loading_cell_subtype_weights_per_CM_standard_scale_col_clustermap.pdf",
    bbox_inches="tight",
    dpi=300,
)
plt.close(g.fig)


# 3. Tumor-centric CM node plot


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D


BASE_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_analysis")
OUTPUT_DIR = BASE_DIR / "balanced_joint_nmf_outputs"
JOINT_CM_DIR = OUTPUT_DIR / "joint_cm"
NETWORK_DIR = JOINT_CM_DIR / "networks"
TABLE_DIR = JOINT_CM_DIR / "tables"
SHARED_DIR = OUTPUT_DIR / "shared"
FIGURE_DIR = JOINT_CM_DIR / "figures"

EDGE_TABLE = NETWORK_DIR / "status_specific_nodeplot_edges.csv"
NODE_TABLE = NETWORK_DIR / "tumor_network_nodes_from_H_df.csv"

OUT_STEM = "tumor_centric_nodeplot_edge_origin"
EDGE_THRESHOLD = 0.25

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


def prefix_color_map_from_nodes(nodes: list[str]) -> dict[str, tuple[float, float, float]]:
    prefixes = sorted({node.split("_")[0] for node in nodes})
    palette = plt.get_cmap("tab20").colors
    return {prefix: palette[i % len(palette)] for i, prefix in enumerate(prefixes)}


def save_colorbar(cmap, norm, path_stem: Path, label: str) -> None:
    fig, ax = plt.subplots(figsize=(4.0, 0.45))
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=ax, orientation="horizontal")
    cb.set_label(label, fontsize=10)
    cb.ax.tick_params(labelsize=8)
    fig.savefig(path_stem.with_suffix(".pdf"), bbox_inches="tight", dpi=300)
    fig.savefig(path_stem.with_suffix(".svg"), bbox_inches="tight", dpi=300)
    plt.close(fig)


def save_edge_class_legend(path_stem: Path) -> None:
    handles = [
        Line2D([0], [0], color="#1d4ed8", lw=3, label="Tumor only"),
        Line2D([0], [0], color="#b91c1c", lw=3, label="Shared with normal-like"),
    ]
    fig, ax = plt.subplots(figsize=(3.8, 1.1))
    ax.axis("off")
    ax.legend(handles=handles, loc="center", frameon=False, ncol=2)
    fig.savefig(path_stem.with_suffix(".pdf"), bbox_inches="tight", dpi=300)
    fig.savefig(path_stem.with_suffix(".svg"), bbox_inches="tight", dpi=300)
    plt.close(fig)








def edge_key(node_a: str, node_b: str) -> tuple[str, str]:
    return tuple(sorted((node_a, node_b)))


def plot_tumor_centric_nodeplot() -> None:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    edges = pd.read_csv(EDGE_TABLE)
    nodes = pd.read_csv(NODE_TABLE)
    nodes = nodes.sort_values(["CM", "rank"])

    tumor_edges = edges.loc[edges["context"].eq("tumor")].copy()
    normal_edges = edges.loc[edges["context"].eq("normal-like")].copy()

    normal_pass = {
        (row.CM, edge_key(row.node_a, row.node_b)): bool(row["edge_pass_r_ge_0.25"])
        for _, row in normal_edges.iterrows()
    }

    cm_names = nodes["CM"].drop_duplicates().tolist()
    all_nodes = nodes["node"].tolist()
    prefix_colors = prefix_color_map_from_nodes(all_nodes)

    shared_cmap = LinearSegmentedColormap.from_list("shared_red", ["#fee2e2", "#b91c1c"])
    tumor_only_cmap = LinearSegmentedColormap.from_list("tumor_only_blue", ["#dbeafe", "#1d4ed8"])
    edge_norm = Normalize(vmin=EDGE_THRESHOLD, vmax=1.0)

    n_cols = 2
    n_rows = int(np.ceil(len(cm_names) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.4, n_rows * 4.2), squeeze=False)
    axes = axes.flatten()

    summary = []
    for ax, cm_name in zip(axes, cm_names):
        cm_nodes = nodes.loc[nodes["CM"].eq(cm_name), "node"].tolist()
        cm_tumor_edges = tumor_edges.loc[tumor_edges["CM"].eq(cm_name)].copy()
        cm_tumor_edges = cm_tumor_edges.loc[cm_tumor_edges["edge_pass_r_ge_0.25"].astype(bool)]

        graph = nx.Graph()
        graph.add_nodes_from(cm_nodes)

        shared_count = 0
        tumor_only_count = 0
        for _, row in cm_tumor_edges.iterrows():
            pair = edge_key(row.node_a, row.node_b)
            is_shared = normal_pass.get((cm_name, pair), False)
            edge_class = "shared" if is_shared else "tumor_only"
            if is_shared:
                shared_count += 1
            else:
                tumor_only_count += 1
            graph.add_edge(row.node_a, row.node_b, weight=float(row.pearson_r), edge_class=edge_class)

        summary.append({"CM": cm_name, "shared_edges": shared_count, "tumor_only_edges": tumor_only_count})

        ax.set_title(cm_name, fontsize=14, fontweight="bold")
        ax.axis("off")
        if not cm_nodes:
            ax.text(0.5, 0.5, "No retained nodes", ha="center", va="center", fontsize=10)
            continue

        pos = nx.circular_layout(cm_nodes)

        if graph.number_of_edges() > 0:
            for edge_class, cmap in [("shared", shared_cmap), ("tumor_only", tumor_only_cmap)]:
                class_edges = [(a, b, d) for a, b, d in graph.edges(data=True) if d["edge_class"] == edge_class]
                if not class_edges:
                    continue
                nx.draw_networkx_edges(
                    graph,
                    pos,
                    edgelist=[(a, b) for a, b, _ in class_edges],
                    edge_color=[cmap(edge_norm(d["weight"])) for _, _, d in class_edges],
                    width=[2.0 + 3.0 * edge_norm(d["weight"]) for _, _, d in class_edges],
                    alpha=0.88,
                    ax=ax,
                )

        node_colors = [prefix_colors.get(node.split("_")[0], "#999999") for node in cm_nodes]
        nx.draw_networkx_nodes(
            graph,
            pos,
            nodelist=cm_nodes,
            node_color=node_colors,
            node_size=1700,
            linewidths=0.8,
            edgecolors="white",
            ax=ax,
        )
        nx.draw_networkx_labels(
            graph,
            pos,
            labels={node: node for node in cm_nodes},
            font_size=8,
            font_color="black",
            ax=ax,
        )

    for ax in axes[len(cm_names):]:
        ax.axis("off")

    fig.suptitle(
        "Tumor-centric CM node plots: tumor-only vs shared edges",
        y=1.005,
        fontweight="bold",
        fontsize=16,
    )
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{OUT_STEM}.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(FIGURE_DIR / f"{OUT_STEM}.svg", bbox_inches="tight", dpi=300)
    plt.close(fig)

    save_colorbar(
        tumor_only_cmap,
        edge_norm,
        FIGURE_DIR / f"{OUT_STEM}_tumor_only_edge_colorbar",
        "Tumor-only edge correlation in tumor samples (r)",
    )
    save_colorbar(
        shared_cmap,
        edge_norm,
        FIGURE_DIR / f"{OUT_STEM}_shared_edge_colorbar",
        "Shared edge correlation in tumor samples (r)",
    )
    save_edge_class_legend(FIGURE_DIR / f"{OUT_STEM}_edge_class_legend")

    summary_df = pd.DataFrame(summary)
    print(summary_df.to_string(index=False))
    print("Total shared edges:", int(summary_df["shared_edges"].sum()))
    print("Total tumor-only edges:", int(summary_df["tumor_only_edges"].sum()))
    print("Saved:", FIGURE_DIR / f"{OUT_STEM}.pdf")
    print("Saved:", FIGURE_DIR / f"{OUT_STEM}.svg")

plot_tumor_centric_nodeplot()


# 4. W_df and epithelial-frequency Spearman scatter plots


In [ ]:
import re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["svg.fonttype"] = "none"
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

sns.set_style("ticks")

BASE_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_analysis")
INPUT_DIR = BASE_DIR / "balanced_joint_nmf_outputs"
OUT_DIR = BASE_DIR / "W_df_epi_frequency_scatter_plots_spearman_only"
TUMOR_DIR = OUT_DIR / "tumor"
NORMAL_DIR = OUT_DIR / "normal_like"
for d in [OUT_DIR, TUMOR_DIR, NORMAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

w_df = pd.read_csv(INPUT_DIR / "joint_cm" / "tables" / "W_df.csv", index_col=0)
epi_freq = pd.read_csv(INPUT_DIR / "shared" / "epi_subtype_frequency.csv", index_col=0)
sample_status = pd.read_csv(INPUT_DIR / "shared" / "sample_status.csv", index_col=0)

COLOR_MAP_PATH = BASE_DIR / "adata_celltype" / "umap_cell_subtype_ref_colors_publication" / "reference_cell_subtype_color_map.csv"
color_map_df = pd.read_csv(COLOR_MAP_PATH)
epi_color_map = dict(zip(color_map_df["cell_subtype"], color_map_df["color"]))
default_epi_color = "#4C72B0"

common_samples = w_df.index.intersection(epi_freq.index).intersection(sample_status.index)
w_df = w_df.loc[common_samples]
epi_freq = epi_freq.loc[common_samples]
sample_status = sample_status.loc[common_samples]

tumor_samples = sample_status.index[sample_status["status"] == "tumor"]
normal_samples = sample_status.index[sample_status["status"] == "normal-like"]

print(f"W_df shape: {w_df.shape}")
print(f"epi_freq shape: {epi_freq.shape}")
print(f"tumor samples: {len(tumor_samples)}")
print(f"normal-like samples: {len(normal_samples)}")


def safe_filename_part(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")


def safe_spearman(x, y):
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)
    valid = x.notna() & y.notna()
    x = x.loc[valid]
    y = y.loc[valid]
    if len(x) < 3 or x.nunique() <= 1 or y.nunique() <= 1:
        return np.nan, np.nan
    return spearmanr(x, y)


def fmt_num(x, digits=3):
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"


def fmt_p(x):
    if pd.isna(x):
        return "NA"
    return f"{x:.1e}" if x < 0.001 else f"{x:.3f}"


def plot_scatter_for_group(sample_type, sample_label, samples, out_dir):
    results = []
    for cm_name in w_df.columns:
        for epi_key in epi_freq.columns:
            plot_df = pd.DataFrame(
                {
                    "CM_score": w_df.loc[samples, cm_name],
                    "Epi_fraction": epi_freq.loc[samples, epi_key],
                },
                index=samples,
            ).dropna()

            sr, p_s = safe_spearman(plot_df["CM_score"], plot_df["Epi_fraction"])
            epi_color = epi_color_map.get(epi_key, default_epi_color)

            sns.set_style("ticks")
            plt.figure(figsize=(6, 5))
            ax = sns.regplot(
                data=plot_df,
                x="CM_score",
                y="Epi_fraction",
                color=epi_color,
                scatter_kws={"s": 60, "alpha": 0.85, "edgecolor": "none"},
                line_kws={"lw": 2},
            )

            ax.set_xlabel(f"{cm_name} score")
            ax.set_ylabel(f"{epi_key} fraction")
            ax.set_title(f"{sample_label}: {cm_name} vs {epi_key}")
            ax.set_ylim(0, 1)

            txt = f"Spearman rho={fmt_num(sr)}, p={fmt_p(p_s)}\nn={len(plot_df)}"
            ax.text(
                0.02,
                0.98,
                txt,
                transform=ax.transAxes,
                va="top",
                ha="left",
                fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
            )

            sns.despine()
            plt.tight_layout()

            stem = f"scatter_{safe_filename_part(cm_name)}_vs_{safe_filename_part(epi_key)}"
            out_png = out_dir / f"{stem}.png"
            out_pdf = out_dir / f"{stem}.pdf"
            plt.savefig(out_png, dpi=300, bbox_inches="tight")
            plt.savefig(out_pdf, dpi=300, bbox_inches="tight")
            plt.close()

            results.append(
                {
                    "sample_type": sample_type,
                    "CM": cm_name,
                    "epi_subtype": epi_key,
                    "n": len(plot_df),
                    "spearman_rho": sr,
                    "spearman_p": p_s,
                    "epi_color": epi_color,
                    "png": str(out_png),
                    "pdf": str(out_pdf),
                }
            )

    return pd.DataFrame(results)


tumor_results = plot_scatter_for_group("tumor", "Tumor samples", tumor_samples, TUMOR_DIR)
normal_results = plot_scatter_for_group("normal-like", "Normal-like samples", normal_samples, NORMAL_DIR)
summary_df = pd.concat([tumor_results, normal_results], ignore_index=True)
summary_df.to_csv(OUT_DIR / "W_df_epi_frequency_scatter_spearman_only_summary.csv", index=False)

print(f"Saved tumor figures to: {TUMOR_DIR}")
print(f"Saved normal-like figures to: {NORMAL_DIR}")
print(f"Saved summary to: {OUT_DIR / 'W_df_epi_frequency_scatter_spearman_only_summary.csv'}")
print(f"Total scatter pairs: {summary_df.shape[0]}")


# 5. W_df Min–Max clustermap with series annotation


# W_df Min-Max Activity Clustermap With Series Annotation

This notebook reproduces the `w_df_activity_sample_activity_per_CM_standard_scale_col_clustermap` plotting style from `balanced_joint_nmf_cm_decomposition.ipynb`, using the already min-max scaled table `w_df_activity_sample_by_CM_standard_scale_col.csv`.

The only intentional addition is an extra row annotation from `adata_anno_cell_subtype_re.h5ad.obs["series"]`.

In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

BASE_DIR = Path('/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_analysis')
TABLE_DIR = BASE_DIR / 'balanced_joint_nmf_outputs' / 'joint_cm' / 'tables'
SHARED_DIR = BASE_DIR / 'balanced_joint_nmf_outputs' / 'shared'
FIG_DIR = BASE_DIR / 'balanced_joint_nmf_outputs' / 'joint_cm' / 'figures'
ADATA_PATH = BASE_DIR / 'adata_anno_cell_subtype_re.h5ad'

W_MINMAX_PATH = TABLE_DIR / 'w_df_activity_sample_by_CM_standard_scale_col.csv'
SAMPLE_STATUS_PATH = SHARED_DIR / 'sample_status.csv'

OUT_STEM = FIG_DIR / 'w_df_activity_sample_activity_per_CM_standard_scale_col_with_series_clustermap'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('W min-max:', W_MINMAX_PATH)
print('sample status:', SAMPLE_STATUS_PATH)
print('adata:', ADATA_PATH)
print('output stem:', OUT_STEM)

In [ ]:
def setup_plot_style() -> None:
    mpl.rcParams['pdf.fonttype'] = 42
    mpl.rcParams['ps.fonttype'] = 42
    mpl.rcParams['svg.fonttype'] = 'none'
    mpl.rcParams['font.family'] = 'Arial'
    mpl.rcParams['axes.labelcolor'] = 'black'
    mpl.rcParams['xtick.color'] = 'black'
    mpl.rcParams['ytick.color'] = 'black'
    mpl.rcParams['text.color'] = 'black'
    sns.set_theme(style='white', font='Arial')

setup_plot_style()

In [ ]:
w_minmax = pd.read_csv(W_MINMAX_PATH, index_col=0)
sample_status = pd.read_csv(SAMPLE_STATUS_PATH, index_col=0)

adata = ad.read_h5ad(ADATA_PATH, backed='r')
required_obs = {'sample', 'series'}
missing = required_obs.difference(adata.obs.columns)
if missing:
    raise KeyError(f'adata.obs missing required columns: {sorted(missing)}')

sample_series = adata.obs[['sample', 'series']].dropna().drop_duplicates().copy()
series_nunique = sample_series.groupby('sample')['series'].nunique()
ambiguous_samples = series_nunique[series_nunique > 1]
if not ambiguous_samples.empty:
    raise ValueError(f'Samples with multiple series values: {ambiguous_samples.to_dict()}')

series_by_sample = sample_series.drop_duplicates('sample').set_index('sample')['series'].astype(str)

samples = sample_status.index.intersection(w_minmax.index).intersection(series_by_sample.index)
plot_df = w_minmax.loc[samples].copy()
status_series = sample_status.loc[samples, 'status'].astype(str)
series_series = series_by_sample.loc[samples].astype(str)

print('plot_df shape:', plot_df.shape)
print('status counts:')
print(status_series.value_counts())
print('series counts:')
print(series_series.value_counts())
print('missing W samples without series:', sorted(set(w_minmax.index) - set(series_by_sample.index)))

In [ ]:
status_palette = {'normal-like': '#377EB8', 'tumor': '#E41A1C'}
series_values = sorted(series_series.dropna().unique())
# Use colors intentionally different from the tumor/normal status colors.
series_color_list = [
    '#1B9E77',  # green
    '#D95F02',  # orange
    '#7570B3',  # purple
    '#66A61E',  # olive green
    '#E6AB02',  # gold
    '#A6761D',  # brown
    '#666666',  # gray
    '#E7298A',  # magenta
    '#00897B',
    '#7B3294',
]
if len(series_values) > len(series_color_list):
    raise ValueError(f'Need more series colors: {len(series_values)} series values')
series_palette = {series: series_color_list[i] for i, series in enumerate(series_values)}

row_colors = pd.DataFrame(
    {
        'Status': status_series.map(status_palette).fillna('#999999'),
        'Series': series_series.map(series_palette).fillna('#999999'),
    },
    index=plot_df.index,
)

figsize = (plot_df.shape[1] * 0.35 + 2.5, plot_df.shape[0] * 0.13 + 2.5)

g = sns.clustermap(
    plot_df,
    cmap='viridis',
    vmin=0,
    vmax=1,
    center=None,
    cbar_kws={'label': 'Column min-max activity'},
    col_cluster=False,
    figsize=figsize,
    row_colors=row_colors,
    colors_ratio=(0.06, 0.01),
)

g.fig.suptitle('W_df activity: sample activity per CM (column min-max, with series)', y=1.02)
ax = g.ax_heatmap
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_yticklabels([])
ax.set_yticks([])
ax.tick_params(left=False)
ax.set_ylabel('')

status_handles = [mpl.patches.Patch(facecolor=color, label=label) for label, color in status_palette.items()]
series_handles = [mpl.patches.Patch(facecolor=color, label=label) for label, color in series_palette.items()]
status_legend = g.ax_col_dendrogram.legend(
    handles=status_handles,
    title='Status',
    loc='upper left',
    bbox_to_anchor=(1.01, 1.0),
    frameon=False,
)
g.ax_col_dendrogram.add_artist(status_legend)
g.ax_col_dendrogram.legend(
    handles=series_handles,
    title='Series',
    loc='upper left',
    bbox_to_anchor=(1.01, 0.55),
    frameon=False,
)

g.fig.tight_layout(rect=[0, 0, 1, 0.98])
g.savefig(OUT_STEM.with_suffix('.pdf'), bbox_inches='tight', dpi=300)
g.savefig(OUT_STEM.with_suffix('.svg'), bbox_inches='tight', dpi=300)
plt.close(g.fig)

plot_df.to_csv(TABLE_DIR / 'w_df_activity_sample_by_CM_standard_scale_col_with_series_plot_matrix.csv')
pd.DataFrame({'status': status_series, 'series': series_series}).to_csv(TABLE_DIR / 'w_df_activity_sample_by_CM_standard_scale_col_with_series_row_annotations.csv')

print('saved:', OUT_STEM.with_suffix('.pdf'))
print('saved:', OUT_STEM.with_suffix('.svg'))
print('saved matrix:', TABLE_DIR / 'w_df_activity_sample_by_CM_standard_scale_col_with_series_plot_matrix.csv')
print('saved annotations:', TABLE_DIR / 'w_df_activity_sample_by_CM_standard_scale_col_with_series_row_annotations.csv')